In [7]:
import pandas as pd
import numpy as np
import os


# 1. FUNÇÃO DE LEITURA E UNIÃO

def merge_excel_sheets_to_dataframe(directory_path):
    dataframes = []
    print("Iniciando a leitura e união das planilhas...")
    for filename in os.listdir(directory_path):
        if filename.endswith('.xlsx') or filename.endswith('.xls'):
            file_path = os.path.join(directory_path, filename)
            try:
                excel_file = pd.ExcelFile(file_path)
                for sheet_name in excel_file.sheet_names:
                    df = pd.read_excel(file_path, sheet_name=sheet_name)
                    df['Source_File'] = filename
                    df['Sheet_Name'] = sheet_name
                    dataframes.append(df)
            except Exception as e:
                print(f"Erro ao ler o arquivo {filename}: {e}")
    
    if not dataframes:
        return pd.DataFrame()
        
    merged_dataframe = pd.concat(dataframes, ignore_index=True)
    print(f"\nUnião completa! Total de {len(merged_dataframe)} registros lidos brutos.")
    return merged_dataframe


# 2. FUNÇÃO DE IMPUTAÇÃO COM COTA LIMITADA

def conditional_imputation_with_quota(df, col_ig, cols_early, cols_late, cut_off_weeks=20, quota_pct=0.025):
    df_processed = df.copy()
    df_processed['row_is_virtual'] = 0
    total_rows = len(df_processed)
    max_changes_per_group = int(total_rows * quota_pct)
    
    print(f"\n--- INICIANDO IMPUTAÇÃO (COTA MÁXIMA DE {quota_pct*100}% POR GRUPO) ---")
    
    df_processed[col_ig] = pd.to_numeric(df_processed[col_ig], errors='coerce')
    mask_early_group = df_processed[col_ig] < cut_off_weeks
    mask_late_group = df_processed[col_ig] >= cut_off_weeks

    # Grupo Early (< 20 semanas)
    missing_early_mask = mask_early_group & (df_processed[cols_early].isnull().any(axis=1))
    candidates_early_idx = df_processed[missing_early_mask].index.tolist()
    
    if len(candidates_early_idx) > max_changes_per_group:
        selected_early_idx = np.random.choice(candidates_early_idx, size=max_changes_per_group, replace=False)
    else:
        selected_early_idx = candidates_early_idx

    if len(selected_early_idx) > 0:
        for col in cols_early:
            median_val = df_processed.loc[mask_early_group, col].median()
            df_processed.loc[selected_early_idx, col] = df_processed.loc[selected_early_idx, col].fillna(median_val)
        df_processed.loc[selected_early_idx, 'row_is_virtual'] = 1

    # Grupo Late (>= 20 semanas)
    missing_late_mask = mask_late_group & (df_processed[cols_late].isnull().any(axis=1))
    candidates_late_idx = df_processed[missing_late_mask].index.tolist()
    
    if len(candidates_late_idx) > max_changes_per_group:
        selected_late_idx = np.random.choice(candidates_late_idx, size=max_changes_per_group, replace=False)
    else:
        selected_late_idx = candidates_late_idx
        
    if len(selected_late_idx) > 0:
        for col in cols_late:
            median_val = df_processed.loc[mask_late_group, col].median()
            df_processed.loc[selected_late_idx, col] = df_processed.loc[selected_late_idx, col].fillna(median_val)
        df_processed.loc[selected_late_idx, 'row_is_virtual'] = 1
            
    return df_processed


# 3. REMOVE LINHAS VAZIAS

def drop_incomplete_rows(df, col_ig, cols_early, cols_late, cut_off=20):
    print("\n--- LIMPANDO DADOS INCOMPLETOS RESTANTES ---")
    
    df_clean = df.copy()
    df_clean[col_ig] = pd.to_numeric(df_clean[col_ig], errors='coerce')
    
    df_clean = df_clean.dropna(subset=[col_ig])
    
    fail_early = (df_clean[col_ig] < cut_off) & (df_clean[cols_early].isnull().any(axis=1))
    fail_late = (df_clean[col_ig] >= cut_off) & (df_clean[cols_late].isnull().any(axis=1))
    
    rows_to_drop_mask = fail_early | fail_late
    df_final = df_clean[~rows_to_drop_mask]
    
    print(f"Linhas mantidas após limpeza: {len(df_final)} de {len(df)}")
    return df_final


# 4. NOVA FUNÇÃO: FILTRO LONGITUDINAL

def filter_longitudinal_patients(df, pid_col, ig_col, cut_off_weeks=20):
    print("\n--- FILTRO LONGITUDINAL: EXIGINDO EXAMES ANTES E DEPOIS DE 20 SEMANAS ---")
    
    df_temp = df.copy()
    df_temp[ig_col] = pd.to_numeric(df_temp[ig_col], errors='coerce')
    df_temp = df_temp.dropna(subset=[pid_col, ig_col])
    
    initial_patients = df_temp[pid_col].nunique()
    initial_rows = len(df_temp)
    
    pids_early = set(df_temp[df_temp[ig_col] < cut_off_weeks][pid_col])
    pids_late = set(df_temp[df_temp[ig_col] >= cut_off_weeks][pid_col])
    
    # Interseção: O paciente precisa existir nos dois grupos
    valid_pids = pids_early.intersection(pids_late)
    
    df_filtered = df[df[pid_col].isin(valid_pids)].copy()
    
    final_patients = df_filtered[pid_col].nunique()
    
    print(f"Pacientes únicos antes do filtro: {initial_patients}")
    print(f"Pacientes válidos (Longitudinais): {final_patients}")
    print(f"Pacientes descartados (Sem acompanhamento completo): {initial_patients - final_patients}")
    print(f"Total de linhas resultantes: {len(df_filtered)}")
    
    return df_filtered, valid_pids


# SCRIPT PRINCIPAL


# Caminho da pasta (Ajuste se necessário)
directory_path = r'C:\Users\Usuario\Jupyter\CIUR\04-03-26\Data Treatment\planilhas'

PID_COL = 'PID'
IG_COL = 'IG no exame'
EARLY_COLS = ['CCN']
LATE_COLS = ['CC', 'CF', 'DBP']
DATA_COLS = EARLY_COLS + LATE_COLS 
OTHER_COLS = [' (4) (2)', 'IG exame'] 

# 1. Ler planilhas
merged_df = merge_excel_sheets_to_dataframe(directory_path)

if not merged_df.empty:
    if PID_COL not in merged_df.columns:
        merged_df[PID_COL] = np.nan

    # 2. Imputação Controlada
    df_imputed = conditional_imputation_with_quota(
        merged_df, IG_COL, EARLY_COLS, LATE_COLS, quota_pct=0.025
    )

    # 3. Remover vazios restantes (Strict Cleaning)
    df_cleaned = drop_incomplete_rows(
        df_imputed, IG_COL, EARLY_COLS, LATE_COLS, cut_off=20
    )

    # 4. Ordenação e Deduplicação Básica (Mesmo Paciente + Mesma IG)
    sort_cols = DATA_COLS + OTHER_COLS + [IG_COL]
    existing_sort_cols = [c for c in sort_cols if c in df_cleaned.columns]
    
    df_sorted = df_cleaned.sort_values(by=existing_sort_cols, ascending=False, ignore_index=True)
    df_sorted['is_duplicate'] = df_sorted.duplicated(subset=[PID_COL, IG_COL], keep='first')
    
    accepted_df = df_sorted[~df_sorted['is_duplicate']].copy()
    unaccepted_duplicates = df_sorted[df_sorted['is_duplicate']].copy()

    # 5. Filtro Longitudinal (A grande peneira final)
    accepted_longitudinal_df, valid_pids = filter_longitudinal_patients(
        accepted_df, PID_COL, IG_COL, cut_off_weeks=20
    )

    # Identificar as linhas que foram rejeitadas pelo filtro longitudinal para adicionar ao log de lixo
    discarded_longitudinal_df = accepted_df[~accepted_df[PID_COL].isin(valid_pids)].copy()
    
    # Unir todos os descartados (duplicatas + sem acompanhamento) no arquivo de log
    final_unaccepted_df = pd.concat([unaccepted_duplicates, discarded_longitudinal_df], ignore_index=True)

    # 6. Relatório Final
    print("\n" + "="*60)
    print("RESUMO FINAL: ")
    print("="*60)
    total_rows = len(accepted_longitudinal_df)
    virtual_rows = accepted_longitudinal_df['row_is_virtual'].sum() if 'row_is_virtual' in accepted_longitudinal_df.columns else 0
    real_rows = total_rows - virtual_rows
    
    pct_real = (real_rows / total_rows * 100) if total_rows > 0 else 0
    pct_virtual = (virtual_rows / total_rows * 100) if total_rows > 0 else 0

    print(f"TOTAL DE LINHAS PARA O MODELO: {total_rows}")
    print(f"TOTAL DE PACIENTES ÚNICOS: {accepted_longitudinal_df[PID_COL].nunique()}")
    print(f"LINHAS REAIS:    {real_rows} ({pct_real:.2f}%)")
    print(f"LINHAS VIRTUAIS: {virtual_rows} ({pct_virtual:.2f}%)")
    print("="*60)

    # 7. Salvar Arquivos
    cols_to_save = [PID_COL] + existing_sort_cols + ['row_is_virtual']
    cols_to_save = list(dict.fromkeys(cols_to_save)) # Remove duplicatas na lista de colunas
    cols_to_save = [c for c in cols_to_save if c in accepted_longitudinal_df.columns]
    
    accepted_longitudinal_df[cols_to_save].to_csv('accepted_PIDS_final_para_modelo.csv', index=False)
    final_unaccepted_df.to_csv('unaccepted_PIDS_duplicates_log.csv', index=False)
    
    print(f"\nArquivos salvos com sucesso! O 'accepted_PIDS_final_para_modelo.csv' está pronto para Machine Learning.")
else:
    print("O processo foi abortado porque nenhuma planilha pôde ser lida.")

Iniciando a leitura e união das planilhas...

União completa! Total de 563886 registros lidos brutos.

--- INICIANDO IMPUTAÇÃO (COTA MÁXIMA DE 2.5% POR GRUPO) ---

--- LIMPANDO DADOS INCOMPLETOS RESTANTES ---
Linhas mantidas após limpeza: 364154 de 563886

--- FILTRO LONGITUDINAL: EXIGINDO EXAMES ANTES E DEPOIS DE 20 SEMANAS ---
Pacientes únicos antes do filtro: 92326
Pacientes válidos (Longitudinais): 36499
Pacientes descartados (Sem acompanhamento completo): 55827
Total de linhas resultantes: 201416

RESUMO FINAL: 
TOTAL DE LINHAS PARA O MODELO: 201416
TOTAL DE PACIENTES ÚNICOS: 36499
LINHAS REAIS:    186523 (92.61%)
LINHAS VIRTUAIS: 14893 (7.39%)

Arquivos salvos com sucesso! O 'accepted_PIDS_final_para_modelo.csv' está pronto para Machine Learning.
